# Lab 8 — Bidirectional Chat with WebSockets

**Difficulty: Intermediate | ~40 min | Requires Lab 2 (Async/Await) and Lab 7 (Streaming Responses with SSE)**

### Step 0: Install Dependencies

Every pinned dependency in one line. `uvicorn` is required so the `TestClient` can serve the WebSocket route.

In [2]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0 uvicorn==0.30.6


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

The API key is loaded from `.env` with an input fallback. `TestClient` from Starlette is FastAPI's recommended way to test WebSocket endpoints without running a full uvicorn server. `collections.deque` will hold queued messages that arrive while generation is already in progress.

In [3]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient
from dotenv import load_dotenv
from openai import AsyncOpenAI
from collections import deque
import asyncio, os

load_dotenv()

api_key = os.getenv("OPEN_ROUTER_KEY")
if not api_key:
    api_key = input("Open Router API key: ")

client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

app = FastAPI()

### Step 2: The `generate()` Coroutine

`generate()` takes the open WebSocket and the conversation history, opens a streaming completion against OpenRouter, forwards each `delta.content` token to the client as a `token` event, and accumulates the full answer. The `try/finally` guarantees the upstream stream is always closed — even if the task is cancelled mid-stream by a `stop` signal. It returns the full assembled answer, which the orchestrator (Step 5) saves into history.

`client` is a module-level global defined in Step 1, so the function can reference it directly. Note it does **not** mutate `history` itself — it only reads it for context and returns the new text, keeping each function's responsibility narrow.

In [4]:
async def generate(websocket, history):
    stream = await client.chat.completions.create(
        model="openrouter/free",
        messages=history,
        stream=True,
    )
    full_answer = ""
    try:
        async for chunk in stream:
            delta = chunk.choices[0].delta
            if delta.content:
                full_answer += delta.content
                await websocket.send_json({"type": "token", "content": delta.content})
    finally:
        await stream.close()
    return full_answer

### Step 3: The `handle_message()` Coroutine

When a second message arrives while generation is already running, the server must not interrupt the answer. `handle_message()` appends that message to the pending queue and immediately acknowledges it to the client with a `queued` event. It does not touch `history` and it does not cancel anything — generation continues uninterrupted.

In [5]:
async def handle_message(websocket, data, pending_messages):
    pending_messages.append(data)
    await websocket.send_json({"type": "queued", "content": data["content"]})

### Step 4: The `handle_stop()` Coroutine

A `stop` signal is the opposite of a queued message: it cancels the in-flight generation task and tells the client with a `cancelled` event. Because `generate()` wraps its stream in `try/finally`, cancelling the task also closes the upstream HTTP connection. `handle_stop()` deliberately does not save any partial answer to history.

In [6]:
async def handle_stop(websocket, gen_task):
    gen_task.cancel()
    await websocket.send_json({"type": "cancelled"})

### Step 5: The `websocket_chat()` Orchestrator

The orchestrator is the route handler. It stays alive for the entire conversation, so `history` and `pending_messages` are plain local variables that persist across every turn without any external store, because the function never returns until the connection closes. FastAPI requires the parameter to be annotated with the `WebSocket` type — without it, FastAPI cannot recognize the argument as the WebSocket connection and rejects the upgrade.

**Outer loop:** either pop a queued message from `pending_messages` or wait for the next incoming message from `websocket.receive_json()`. Queued messages are processed before the handler waits for fresh input — that is how a `queued` message becomes the next turn automatically.

**Generation + receive race:** when a `message` arrives, the orchestrator starts two concurrent tasks: `generate()` to stream tokens, and a receive task awaiting the next client message. `asyncio.wait` with `return_when=asyncio.FIRST_COMPLETED` returns as soon as either finishes. If generation finishes first, the receive task is cancelled, the answer is saved to history, and a `done` event is sent. If a client message arrives first, its `type` decides: `stop` defers to `handle_stop()` and ends the turn; anything else defers to `handle_message()` and the race restarts with a fresh receive task.

Notice `gen_task = None` is initialized before the loop. The `except WebSocketDisconnect` block references it to cancel any in-flight generation — but the client could disconnect before the first message ever starts generating, so the variable must already exist.

In [7]:
@app.websocket("/ws/chat")
async def websocket_chat(websocket: WebSocket):
    await websocket.accept()
    history = []
    pending_messages = deque()
    gen_task = None

    try:
        while True:
            if pending_messages:
                data = pending_messages.popleft()
            else:
                data = await websocket.receive_json()

            if data["type"] == "message":
                history.append({"role": "user", "content": data["content"]})
                await websocket.send_json({"type": "status", "content": "generating"})

                gen_task = asyncio.create_task(generate(websocket, history))
                recv_task = asyncio.create_task(websocket.receive_json())

                while True:
                    done, _ = await asyncio.wait(
                        {gen_task, recv_task},
                        return_when=asyncio.FIRST_COMPLETED,
                    )

                    if gen_task in done:
                        recv_task.cancel()
                        full_answer = gen_task.result()
                        history.append({"role": "assistant", "content": full_answer})
                        await websocket.send_json({"type": "done", "content": full_answer})
                        break

                    result = recv_task.result()
                    if result["type"] == "stop":
                        await handle_stop(websocket, gen_task)
                        break
                    else:
                        await handle_message(websocket, result, pending_messages)
                        recv_task = asyncio.create_task(websocket.receive_json())

    except WebSocketDisconnect:
        if gen_task and not gen_task.done():
            gen_task.cancel()

### Step 6: Set Up TestClient

`TestClient` wraps the app so we can open a WebSocket connection without running a real uvicorn server. It is created once here and reused by every demo below. Each demo opens a fresh connection with `test_client.websocket_connect("/ws/chat")`.

In [8]:
test_client = TestClient(app)

### Step 7: Run the Demos

Four demos exercise the protocol end to end. Each opens a fresh connection and drives send/receive with `receive_json()`.

#### Demo 1: Normal Single Turn + History Persistence

Two messages on the same open connection. The second answer should reference the first — proving that `history` persisted across turns as a local variable inside the handler.

In [9]:
print("=== Demo 1: Normal turn + history ===")
print()

with test_client.websocket_connect("/ws/chat") as ws:
    ws.send_json({"type": "message", "content": "My favorite color is teal. Remember that."})
    while True:
        msg = ws.receive_json()
        if msg["type"] == "done":
            print(f"Answer 1: {msg['content'][:80]}...")
            break

    ws.send_json({"type": "message", "content": "What is my favorite color?"})
    while True:
        msg = ws.receive_json()
        if msg["type"] == "done":
            print(f"Answer 2: {msg['content'][:80]}...")
            break

=== Demo 1: Normal turn + history ===

Answer 1: Got it! Teal is now in my memory as your favorite color. I'll keep that in mind ...
Answer 2: Your favorite color is **teal**. I've noted that from our conversation. Let me k...


#### Demo 2: Queued Message

Send a message, wait for exactly 2 token events (confirming generation has genuinely started), then send a second message. The server queues it and sends a `queued` acknowledgment immediately. Generation finishes with `done`. Without sending anything further, the queued message is automatically picked up as the next turn — the outer loop finds it in `pending_messages` before waiting for fresh input.

In [10]:
print("=== Demo 2: Queued message ===")
print()

with test_client.websocket_connect("/ws/chat") as ws:
    ws.send_json({"type": "message", "content": "List three planets in our solar system."})
    token_count = 0
    while True:
        msg = ws.receive_json()
        if msg["type"] == "token":
            token_count += 1
            if token_count == 2:
                break
        if msg["type"] in ("done", "status"):
            continue

    ws.send_json({"type": "message", "content": "Now list three moons."})
    while True:
        msg = ws.receive_json()
        if msg["type"] == "queued":
            print(f"Queued ack: {msg['content']!r}")
            break

    while True:
        msg = ws.receive_json()
        if msg["type"] == "done":
            print(f"Turn 1 done: {msg['content'][:60]}...")
            break

    while True:
        msg = ws.receive_json()
        if msg["type"] == "done":
            print(f"Turn 2 (queued) done: {msg['content'][:60]}...")
            break

=== Demo 2: Queued message ===

Queued ack: 'Now list three moons.'
Turn 1 done: Sure! Here are three planets from our solar system:

1. Eart...
Turn 2 (queued) done: Here are three moons in our solar system:

1. **Phobos** – o...


#### Demo 3: Stop Signal

Send a message, wait for 2 token events, then send `{"type": "stop"}`. The server cancels the generation task and sends `cancelled`. No `done` event follows, and the partial answer is never saved to history.

In [ ]:
print("=== Demo 3: Stop signal ===")
print()

with test_client.websocket_connect("/ws/chat") as ws:
    ws.send_json({"type": "message", "content": "Write a very long essay about clouds."})
    token_count = 0
    while True:
        msg = ws.receive_json()
        if msg["type"] == "token":
            token_count += 1
            print(f"token recieved: {msg['content']}")
            if token_count == 2:
                break
        if msg["type"] in ("done", "status"):
            continue

    ws.send_json({"type": "stop"})
    events = []
    while True:
        msg = ws.receive_json()
        events.append(msg)
        if msg["type"] in ("cancelled", "done"):
            break

    types_seen = [e["type"] for e in events]
    print(f"Events after stop: {types_seen}")
    print(f"'cancelled' received: {'cancelled' in types_seen}")
    print(f"'done' received: {'done' in types_seen}")

=== Demo 3: Stop signal ===

token recieved: **
token recieved: The
Events after stop: ['token', 'cancelled']
'cancelled' received: True
'done' received: False


#### Demo 4: Disconnect Cleanup

Open a connection, send a message, wait for the first token to confirm generation has started, then exit the context manager — simulating the client disconnecting mid-generation. This confirms no unhandled error is raised and the in-flight task is cancelled cleanly.

In [12]:
print("=== Demo 4: Disconnect cleanup ===")
print()

try:
    with test_client.websocket_connect("/ws/chat") as ws:
        ws.send_json({"type": "message", "content": "Tell me a very long story."})
        while True:
            msg = ws.receive_json()
            if msg["type"] == "token":
                print("First token received — disconnecting now.")
                break
    print("Context manager exited cleanly — no unhandled error.")
except Exception as e:
    print(f"Unexpected error: {e}")

=== Demo 4: Disconnect cleanup ===

First token received — disconnecting now.
Context manager exited cleanly — no unhandled error.
